In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [2]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import json

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from sklearn.preprocessing import StandardScaler

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 75  # 75 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 30  # 30 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)

        return self

    def clean_data(self):
        # Drop volume if zero or NaN
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        # RSI
        self.df.sort_index(inplace=True)
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_swing_points(self, left_bars=10, right_bars=10):
        # Identify local swing highs/lows
        self.df.sort_index(inplace=True)

        rolling_high = self.df['high'].rolling(left_bars + 1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars + 1, center=False).min()

        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars + 1, center=False).max()
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars + 1, center=False).min()

        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        # Donchian Channels
        self.df.sort_index(inplace=True)

        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Donchian breakout flags
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion measure
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        self.df.sort_index(inplace=True)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        # Historical volatility, Z-score
        self.df.sort_index(inplace=True)

        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)

        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        # Volume-based features if present
        if 'volume' not in self.df.columns:
            return self

        self.df.sort_index(inplace=True)

        # OBV
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spikes
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP approximation
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        # Classify trending vs ranging via ADX threshold
        self.df.sort_index(inplace=True)

        if 'adx' not in self.df.columns:
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        return self

    def add_time_features(self):
        # Hour, day-of-week, cyclical encodings
        self.df.sort_index(inplace=True)

        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        return self

    def add_adaptive_targets_and_stops(self):
        # Adapt targets/stops based on regime
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def label_signals(self):
        """
        Priority is profit first (buy_target/sell_target),
        then stop-loss (buy_sl/sell_sl).
        """
        self.df['Signal'] = 0
        self.df['Entry Price'] = 0.0
        self.df['Exit Price'] = 0.0
        self.df['candles_to_profit'] = 0
        self.df['candles_to_loss'] = 0

        all_idx = self.df.index.tolist()
        n = len(all_idx)

        for i in range(n - 1):
            idx_i = all_idx[i]

            entry_price = self.df.at[idx_i, 'close']
            target = self.df.at[idx_i, 'Target']
            stop_loss = self.df.at[idx_i, 'StopLoss']

            # Hypothetical buy/sell levels
            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss
            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            future_slice = all_idx[i+1:]
            signal_found = False

            for offset, future_idx in enumerate(future_slice, start=1):
                future_high = self.df.at[future_idx, 'high']
                future_low = self.df.at[future_idx, 'low']

                triggers = []
                # Check buy/sell target/SL
                if future_high >= buy_target_price:
                    triggers.append(('buy_target', offset))
                if future_low <= buy_sl_price:
                    triggers.append(('buy_sl', offset))
                if future_low <= sell_target_price:
                    triggers.append(('sell_target', offset))
                if future_high >= sell_sl_price:
                    triggers.append(('sell_sl', offset))

                # If multiple triggers this candle, pick the earliest by priority
                if triggers:
                    priority_map = {
                        'buy_target': 1,
                        'sell_target': 2,
                        'buy_sl': 3,
                        'sell_sl': 4
                    }
                    triggers.sort(key=lambda x: priority_map[x[0]])
                    first_trigger, candle_count = triggers[0]

                    if first_trigger == 'buy_target':
                        self.df.at[idx_i, 'Signal'] = 1
                        self.df.at[idx_i, 'Exit Price'] = buy_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'sell_target':
                        self.df.at[idx_i, 'Signal'] = 3
                        self.df.at[idx_i, 'Exit Price'] = sell_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'buy_sl':
                        self.df.at[idx_i, 'Signal'] = 2
                        self.df.at[idx_i, 'Exit Price'] = buy_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    elif first_trigger == 'sell_sl':
                        self.df.at[idx_i, 'Signal'] = 4
                        self.df.at[idx_i, 'Exit Price'] = sell_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    self.df.at[idx_i, 'Entry Price'] = entry_price
                    signal_found = True
                    break

            # If no trigger in future
            if not signal_found:
                self.df.at[idx_i, 'Signal'] = 0
                self.df.at[idx_i, 'Entry Price'] = entry_price

        return self

    def run_pipeline(self):
        (
            self.preprocess_datetime()
                .clean_data()
                .add_base_timeframe_features()
                .add_swing_points()
                .add_range_breakout_features()
                .add_momentum_features()
                .add_volatility_features()
                .add_volume_features()
                .add_regime_features()
                .add_time_features()
                .add_adaptive_targets_and_stops()
                .label_signals()
        )
        return self

    def get_processed_df(self):
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [9]:
nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

# Fetch candle data for each instrument and each timeframe
nifty_2m = fetch_train_candle_data(10, nf_index_symbol, 2)
nifty_5m = fetch_train_candle_data(10, nf_index_symbol, 5)
nifty_15m = fetch_train_candle_data(10, nf_index_symbol, 15)

bnf_2m = fetch_train_candle_data(10, bnf_index_symbol, 2)
bnf_5m = fetch_train_candle_data(10, bnf_index_symbol, 5)
bnf_15m = fetch_train_candle_data(10, bnf_index_symbol, 15)

print("Fetched historical data for all instruments")

# Clean and remove duplicate datetimes (if any)
nifty_2m = nifty_2m.drop_duplicates(subset='datetime', keep='first')
nifty_5m = nifty_5m.drop_duplicates(subset='datetime', keep='first')
nifty_15m = nifty_15m.drop_duplicates(subset='datetime', keep='first')

bnf_2m = bnf_2m.drop_duplicates(subset='datetime', keep='first')
bnf_5m = bnf_5m.drop_duplicates(subset='datetime', keep='first')
bnf_15m = bnf_15m.drop_duplicates(subset='datetime', keep='first')

# Nifty after processing
nf_train_pipeline_2m = FullFeaturePipeline(nifty_2m)
nf_train_pipeline_2m.run_pipeline()
df_nifty_2m = nf_train_pipeline_2m.get_processed_df()

nf_train_pipeline_5m = FullFeaturePipeline(nifty_5m)
nf_train_pipeline_5m.run_pipeline()
df_nifty_5m = nf_train_pipeline_5m.get_processed_df()

nf_train_pipeline_15m = FullFeaturePipeline(nifty_15m)
nf_train_pipeline_15m.run_pipeline()
df_nifty_15m = nf_train_pipeline_15m.get_processed_df()

# Bank Nifty after processing
bnf_train_pipeline_2m = FullFeaturePipeline(bnf_2m)
bnf_train_pipeline_2m.run_pipeline()
df_bnf_2m = bnf_train_pipeline_2m.get_processed_df()

bnf_train_pipeline_5m = FullFeaturePipeline(bnf_5m)
bnf_train_pipeline_5m.run_pipeline()
df_bnf_5m = bnf_train_pipeline_5m.get_processed_df()

bnf_train_pipeline_15m = FullFeaturePipeline(bnf_15m)
bnf_train_pipeline_15m.run_pipeline()
df_bnf_15m = bnf_train_pipeline_15m.get_processed_df()

Fetched historical data for all instruments


In [10]:
df_bnf_2m.columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'is_swing_high',
       'is_swing_low', 'donchian_high', 'donchian_low', 'donchian_range',
       'donchian_breakout', 'range_expansion', 'stoch_k', 'stoch_d', 'adx',
       'diplus', 'diminus', 'cci', 'log_return', 'hist_vol', 'zscore_return',
       'regime_trend', 'hour', 'day_of_week', 'hour_sin', 'hour_cos',
       'dow_sin', 'dow_cos', 'Target', 'StopLoss', 'Signal',
       'candles_to_profit', 'candles_to_loss'],
      dtype='object')

In [11]:
df_bnf_2m

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss,Signal,candles_to_profit,candles_to_loss
datetime,,,,,,,,,,,,,,,,,,,,,
2022-02-02 10:21:00,39160.60,39177.20,39152.60,39163.20,64.592439,44.142400,0.662614,43.479785,38983.107561,39087.9250,...,2.0,0.500000,-0.866025,0.974928,-0.222521,134.147909,67.073954,2,0,8
2022-02-02 10:23:00,39163.50,39163.50,39135.80,39148.20,61.614329,43.143612,-0.268939,43.412551,38991.884582,39094.4950,...,2.0,0.500000,-0.866025,0.974928,-0.222521,130.182467,65.091233,2,0,7
2022-02-02 10:25:00,39147.60,39153.90,39121.40,39123.10,56.887768,39.867141,-2.836327,42.703469,38995.991219,39097.8050,...,2.0,0.500000,-0.866025,0.974928,-0.222521,127.659441,63.829721,2,0,6
2022-02-02 10:27:00,39124.80,39151.60,39121.10,39130.60,57.926362,37.444070,-4.207519,41.651589,39004.742154,39102.7550,...,2.0,0.500000,-0.866025,0.974928,-0.222521,124.884013,62.442006,2,0,5
2022-02-02 10:29:00,39132.90,39157.00,39125.60,39137.90,58.962628,35.701276,-4.760250,40.461526,39010.505318,39106.9600,...,2.0,0.500000,-0.866025,0.974928,-0.222521,122.541331,61.270666,2,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-29 15:21:00,52327.55,52340.90,52309.65,52324.70,60.649447,35.125877,-5.723796,40.849673,52245.519540,52302.6250,...,1.0,-0.707107,-0.707107,0.781831,0.623490,111.541360,55.770680,2,0,4
2024-10-29 15:23:00,52323.30,52326.00,52311.00,52325.70,60.834395,33.146601,-6.162458,39.309059,52251.738372,52305.9650,...,1.0,-0.707107,-0.707107,0.781831,0.623490,106.788405,53.394203,2,0,2
2024-10-29 15:25:00,52327.25,52332.50,52297.25,52308.05,55.845385,29.810169,-7.599112,37.409281,52260.198438,52308.7100,...,1.0,-0.707107,-0.707107,0.781831,0.623490,106.714234,53.357117,0,0,0


In [12]:
config = {
    "instruments": [
        {
            "name": "NIFTY_15M",
            "df": df_nifty_15m,        # user-defined dataframes
            "lot_size": nf_quantity,   # user-defined lot size
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_15M",
            "df": df_bnf_15m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_5M",
            "df": df_nifty_5m,
            "lot_size": nf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_5M",
            "df": df_bnf_5m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_2M",
            "df": df_nifty_2m,
            "lot_size": nf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_2M",
            "df": df_bnf_2m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
    ],

    "window_size": 5,
    "unrealized_pnl_weight": 0.1,
    "time_penalty_weight": 0.005,
    "volatility_penalty_weight": 0.05,
    "target_bonus": 0.5,
    "sl_penalty": 0.5,
    "misalignment_penalty": 0.5,
    "missed_opportunity_penalty": 0.05,

    # Trailing Stop Parameters
    "enable_trailing_sl": True,
    "trailing_sl_mode": "factor",   # can be "fixed", "percentage", or "factor"
    "trailing_sl_amount": 20.0,
    "trailing_sl_percent": 0.01,
    "trailing_factor": 0.5,

    "initial_cap_multiplier": 10,   # factor for initial capital
    "normalize_data": True          # whether to normalize data for the agent's observation
}

# Margin constants
MARGIN_FRACTION = 0.05
MARGIN_CALL_THRESHOLD = 0.5

In [13]:
class Position:
    # Holds all necessary info about the current position (long or short).
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"             # "flat", "long", or "short"
        self.entry_price = None
        self.sl_points = None            # For initial SL
        self.target_points = None
        self.direction = 0               # +1 for long, -1 for short
        self.quantity = 0
        self.time_in_trade = 0
        self.margin_used = 0.0           # Tracks margin used for the current position

        # Trailing SL related
        self.trailing_sl_price = None    # Absolute trailing SL price
        self.highest_price = None        # For tracking highest price since entry (long)
        self.lowest_price = None         # For tracking lowest price since entry (short)

    def update_unrealized_pnl(self, current_price: float) -> float:
        if self.status == "flat":
            return 0.0
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

        # Initialize trailing stop to the initial SL directly
        if direction == 1:
            self.trailing_sl_price = entry_price - sl_points
            self.highest_price = entry_price
            self.lowest_price = None
        else:
            self.trailing_sl_price = entry_price + sl_points
            self.highest_price = None
            self.lowest_price = entry_price

In [16]:
class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.config = env_config
        self.total_reward = 0.0

        # Window size
        self.window_size = self.config["window_size"]

        # Guidance columns (Signal, candles_to_profit, candles_to_loss)
        self.df_guidance = self.config["df"][['Signal','candles_to_profit','candles_to_loss']].copy()

        # Raw data (needed for environment logic, reward calc, etc.)
        # Dropping the columns that are specifically for guidance
        self.df_raw = self.config["df"].drop(columns=['Signal','candles_to_profit','candles_to_loss']).copy()

        # Decide which columns to exclude from agent's observation
        self.exclude_from_obs = ["StopLoss", "Target"]

        # We'll drop the exclude_from_obs columns from df_raw here
        self.df_obs = self.df_raw.drop(columns=self.exclude_from_obs, errors="ignore").copy()

        self.normalize_data = self.config.get("normalize_data", False)
        if self.normalize_data:
            self.scaler = StandardScaler()
            # Fit only on the columns the agent *will* observe
            self.scaler.fit(self.df_obs.values)

            # Transform the observation DataFrame
            self.df_norm = pd.DataFrame(
                self.scaler.transform(self.df_obs.values),
                columns=self.df_obs.columns,
                index=self.df_obs.index
            )
        else:
            self.df_norm = self.df_obs.copy()

        # Other environment parameters
        self.lot_size = self.config["lot_size"]
        self.transaction_cost = self.config["transaction_cost"]
        self.instrument_name = self.config.get("name", "Unknown")

        # Instead of max(high)*3, use margin-based approach for initial capital
        multiplier = self.config.get("initial_cap_multiplier", 3)
        self.initial_capital = self.df_raw['high'].max() * self.lot_size * MARGIN_FRACTION * multiplier
        self.current_capital = None
        self.current_step = None
        self.position = Position()
        self.trade_log = []

        # Action space: 0 = Hold, 1 = Buy, 2 = Sell, 3 = Close
        self.action_space = spaces.Discrete(4)

        # Prepare observation space based on the shape of df_norm
        n_features = len(self.df_norm.columns)
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32),
            "instrument_info": spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
        })

        # Reward shaping parameters
        self.unrealized_pnl_weight = self.config.get("unrealized_pnl_weight", 0.1)
        self.time_penalty_weight = self.config.get("time_penalty_weight", 0.02)
        self.volatility_penalty_weight = self.config.get("volatility_penalty_weight", 0.1)

        # Guidance reward parameters
        self.target_bonus = self.config.get("target_bonus", 0.5)
        self.sl_penalty = self.config.get("sl_penalty", 0.5)
        self.misalignment_penalty = self.config.get("misalignment_penalty", 0.5)
        self.missed_opportunity_penalty = self.config.get("missed_opportunity_penalty", 0.05)

        # Trailing SL config
        self.enable_trailing_sl = self.config.get("enable_trailing_sl", True)
        self.trailing_sl_mode = self.config.get("trailing_sl_mode", "fixed")
        self.trailing_sl_amount = self.config.get("trailing_sl_amount", 20.0)
        self.trailing_sl_percent = self.config.get("trailing_sl_percent", 0.01)
        self.trailing_factor = self.config.get("trailing_factor", 0.5)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log = []
        self.total_reward = 0.0
        return self._get_observation(), {}

    def step(self, action: int):
        current_data = self.df_raw.iloc[self.current_step]
        guidance_row = self.df_guidance.iloc[self.current_step]
        reward = 0.0

        # -----------------------------
        # Immediate Guidance Reward (Only When Flat)
        # -----------------------------
        if self.position.status == "flat":
            signal = guidance_row['Signal']
            candles_to_profit = guidance_row['candles_to_profit'] if guidance_row['candles_to_profit'] > 0 else 1.0
            candles_to_loss = guidance_row['candles_to_loss'] if guidance_row['candles_to_loss'] > 0 else 1.0

            if signal == 1:  # Target for long
                correct_action = 1
                guidance_reward = self.target_bonus / candles_to_profit
            elif signal == 2:  # SL for long
                correct_action = 3
                guidance_reward = -self.sl_penalty / candles_to_loss
            elif signal == 3:  # Target for short
                correct_action = 2
                guidance_reward = self.target_bonus / candles_to_profit
            elif signal == 4:  # SL for short
                correct_action = 3
                guidance_reward = -self.sl_penalty / candles_to_loss
            else:
                correct_action = None
                guidance_reward = 0.0

            if action == correct_action:
                reward += guidance_reward
            else:
                reward -= self.misalignment_penalty

        # -----------------------------
        # If we have a position, check margin call conditions
        # -----------------------------
        if self.position.status != "flat":
            if self._check_margin_call(current_data['close']):
                reward += self._close_position(current_data, "Margin Call")

        # -----------------------------
        # Update Trailing SL (if enabled and position is open)
        # -----------------------------
        if self.enable_trailing_sl and self.position.status != "flat":
            self._update_trailing_sl(current_data)

        # -----------------------------
        # Check for SL or Trailing SL or Target Hit (intra-candle logic)
        # -----------------------------
        hit_sl, fill_price = self._check_sl_hit(current_data)
        if hit_sl:
            reward += self._close_position(current_data, "SL/Trailing SL Hit", override_exit_price=fill_price)

        # -----------------------------
        # Execute Action
        # -----------------------------
        self._execute_action(action, current_data)

        # -----------------------------
        # Time in trade increment & shaping reward
        # -----------------------------
        if self.position.status != "flat":
            self.position.time_in_trade += 1
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        # -----------------------------
        # Advance environment
        # -----------------------------
        self.current_step += 1
        if self.current_step >= len(self.df_raw) or self.current_capital <= 0:
            done = True
            # Force close if still in position
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")
        else:
            done = False

        self.total_reward += reward
        return self._get_observation(), reward, done, False, {}

    def _get_observation(self) -> dict:
        # Here we provide normalized data to the agent, but raw data is used internally.
        idx = min(self.current_step, len(self.df_norm) - 1)

        # Normalized features
        market_features = self.df_norm.iloc[idx].values.astype(np.float32)

        # Position context
        current_close = self.df_raw.iloc[idx]['close']
        unrealized_pnl = self.position.update_unrealized_pnl(current_close)
        normalized_pnl = unrealized_pnl / self.initial_capital
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            normalized_pnl,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        # History window (normalized)
        history_start = max(0, idx - self.window_size)
        history_slice = self.df_norm.iloc[history_start:idx]
        history = history_slice.values.astype(np.float32)
        if len(history) < self.window_size:
            padding = np.zeros((self.window_size - len(history), self.df_norm.shape[1]), dtype=np.float32)
            history = np.vstack([padding, history])

        # Instrument info
        normalized_lot_size = self.lot_size / 1000.0
        normalized_transaction_cost = self.transaction_cost / 100.0
        instrument_info = np.array([normalized_lot_size, normalized_transaction_cost], dtype=np.float32)

        return {
            "market": market_features,
            "position": position_context,
            "history": history,
            "instrument_info": instrument_info
        }

    def _execute_action(self, action: int, current_data: pd.Series):
        price = current_data['open']

        # Close action
        if action == 3 and self.position.status != "flat":
            self._close_position(current_data, "Manual Close")

        # Buy action
        elif action == 1:
            if self.position.status == "short":
                self._close_position(current_data, "Reverse Close")
            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("long", current_data)

        # Sell action
        elif action == 2:
            if self.position.status == "long":
                self._close_position(current_data, "Reverse Close")
            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("short", current_data)

        # If action == 0 => Hold/do nothing

    def _open_position(self, direction: str, current_data: pd.Series):
        entry_price = current_data['open']
        margin_cost = entry_price * self.lot_size * MARGIN_FRACTION
        self.current_capital -= (margin_cost + self.transaction_cost)
        self.position.margin_used = margin_cost

        self.position.open(
            entry_price=entry_price,
            sl_points=current_data['StopLoss'],
            target_points=current_data['Target'],
            direction=1 if direction == "long" else -1,
            quantity=self.lot_size
        )

        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None,
            "initial_capital": self._format_currency(self.initial_capital),
            "points": None,
            "profit": None,
            "win": None,
            "reward": None,
            "time_in_trade": None
        })

    def _close_position(self, current_data: pd.Series, reason: str, override_exit_price=None) -> float:
        if self.position.status == "flat":
            return 0.0

        direction = self.position.direction
        if override_exit_price is not None:
            exit_price = override_exit_price
        elif "SL" in reason:
            # trailing SL or normal SL
            if self.position.trailing_sl_price is not None:
                exit_price = self.position.trailing_sl_price
            else:
                if direction == 1:
                    exit_price = self.position.entry_price - self.position.sl_points
                else:
                    exit_price = self.position.entry_price + self.position.sl_points
        else:
            # Manual/Force/Reverse => exit on current open
            exit_price = current_data['open']

        realized_pnl = (exit_price - self.position.entry_price) * direction * self.lot_size

        # Return margin + realized PnL - transaction_cost
        margin_return = self.position.margin_used
        self.current_capital += (margin_return + realized_pnl - self.transaction_cost)

        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return

        points = (exit_price - self.position.entry_price) * direction
        formatted_profit = self._format_currency(realized_pnl)
        win = realized_pnl > 0

        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade,
                    "points": round(points, 2),
                    "profit": formatted_profit,
                    "win": win,
                    "reward": total_reward
                })
                break

        self.position.reset()
        return total_reward

    def _check_sl_hit(self, current_data: pd.Series):
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        target_points = self.position.target_points

        # Calculate target price
        if direction == 1:
            target_price = entry_price + target_points
        else:
            target_price = entry_price - target_points

        # If target is broken, update trailing SL up/down to at least the target
        if direction == 1:
            if current_data['high'] >= target_price:
                if self.position.trailing_sl_price is not None:
                    self.position.trailing_sl_price = max(self.position.trailing_sl_price, target_price)
        else:
            if current_data['low'] <= target_price:
                if self.position.trailing_sl_price is not None:
                    self.position.trailing_sl_price = min(self.position.trailing_sl_price, target_price)

        current_open = current_data['open']
        current_high = current_data['high']
        current_low = current_data['low']

        # Decide which SL to use if trailing is set
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if direction == 1:
                effective_sl = entry_price - self.position.sl_points
            else:
                effective_sl = entry_price + self.position.sl_points

        # Check gap scenarios first
        if direction == 1:
            if current_open <= effective_sl:
                return True, current_open
            elif current_low <= effective_sl:
                return True, effective_sl
        else:
            if current_open >= effective_sl:
                return True, current_open
            elif current_high >= effective_sl:
                return True, effective_sl

        return False, None

    def _update_trailing_sl(self, current_data: pd.Series):
        if self.position.direction == 1:  # long
            if self.position.highest_price is None:
                self.position.highest_price = current_data['high']
            else:
                self.position.highest_price = max(self.position.highest_price, current_data['high'])

            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.highest_price,
                side="long"
            )
            initial_sl_price = self.position.entry_price - self.position.sl_points
            self.position.trailing_sl_price = max(
                self.position.trailing_sl_price,
                new_sl_price,
                initial_sl_price
            )
        else:  # short
            if self.position.lowest_price is None:
                self.position.lowest_price = current_data['low']
            else:
                self.position.lowest_price = min(self.position.lowest_price, current_data['low'])

            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.lowest_price,
                side="short"
            )
            initial_sl_price = self.position.entry_price + self.position.sl_points
            self.position.trailing_sl_price = min(
                self.position.trailing_sl_price,
                new_sl_price,
                initial_sl_price
            )

    def _calculate_trailing_sl_price(self, reference_price: float, side: str) -> float:
        if self.trailing_sl_mode == "fixed":
            if side == "long":
                return reference_price - self.trailing_sl_amount
            else:
                return reference_price + self.trailing_sl_amount

        elif self.trailing_sl_mode == "percentage":
            offset = reference_price * self.trailing_sl_percent
            if side == "long":
                return reference_price - offset
            else:
                return reference_price + offset

        elif self.trailing_sl_mode == "factor":
            if side == "long":
                gain = reference_price - self.position.entry_price
                return self.position.entry_price + (gain * self.trailing_factor)
            else:
                gain = self.position.entry_price - reference_price
                return self.position.entry_price - (gain * self.trailing_factor)

        else:
            return reference_price

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        if self.position.sl_points == 0:
            risk_adj_pnl = 0.0
        else:
            risk = self.position.sl_points * self.lot_size
            risk_adj_pnl = (unrealized_pnl / risk) * self.unrealized_pnl_weight

        # Time penalty
        time_penalty = self.time_penalty_weight * (self.position.time_in_trade / 100.0)

        # Volatility penalty
        volatility = (current_data['high'] - current_data['low']) / max(current_data['open'], 1e-6)
        volatility_penalty = self.volatility_penalty_weight * volatility

        # SL proximity penalty
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if self.position.direction == 1:
                effective_sl = self.position.entry_price - self.position.sl_points
            else:
                effective_sl = self.position.entry_price + self.position.sl_points

        if self.position.direction == 1:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, current_data['close'] - effective_sl)
        else:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, effective_sl - current_data['close'])

        if sl_range > 0:
            sl_proximity_ratio = min(sl_distance / sl_range, 1.0)
        else:
            sl_proximity_ratio = 1.0

        sl_proximity_penalty = 0.05 * (1.0 - sl_proximity_ratio)

        total_reward = risk_adj_pnl - time_penalty - volatility_penalty - sl_proximity_penalty
        return float(np.clip(total_reward, -3.0, 3.0))

    def _check_margin_call(self, current_price: float) -> bool:
        if self.position.status == "flat":
            return False

        maintenance_margin = self.position.margin_used * MARGIN_CALL_THRESHOLD
        unrealized_pnl = self.position.update_unrealized_pnl(current_price)
        equity = self.current_capital + unrealized_pnl

        if equity < maintenance_margin:
            return True
        return False

    def get_metrics(self) -> dict:
        if not self.trade_log:
            return {}

        closed_trades = [t for t in self.trade_log if t['exit_time'] is not None]
        total_trades = len(closed_trades)
        if total_trades == 0:
            return {"instrument": self.instrument_name, "message": "No trades taken"}

        winning_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] > 0]
        losing_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] <= 0]

        win_rate = (len(winning_trades) / total_trades) * 100.0
        sum_winning = sum(t['pnl'] for t in winning_trades)
        sum_losing = sum(t['pnl'] for t in losing_trades)
        abs_sum_losing = abs(sum_losing)

        if abs_sum_losing < 1e-10:
            profit_factor = float('inf')
        else:
            profit_factor = sum_winning / abs_sum_losing

        return {
            "instrument": self.instrument_name,
            "initial_capital": self._format_currency(self.initial_capital),
            "final_capital": self._format_currency(self.current_capital),
            "total_trades": total_trades,
            "win_rate": f"{round(win_rate, 2)}%",
            "total_reward": round(self.total_reward, 2),
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": profit_factor
        }

    def _compute_max_drawdown(self) -> float:
        equity_curve = [self.initial_capital]
        for trade in self.trade_log:
            if trade['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + trade['pnl'])
        equity_curve = np.array(equity_curve)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return dd.max()

    def _format_currency(self, value: float) -> str:
        abs_value = abs(value)
        if abs_value >= 1e7:
            return f"₹{value / 1e7:.2f}Cr"
        elif abs_value >= 1e5:
            return f"₹{value / 1e5:.2f}L"
        elif abs_value >= 1e3:
            return f"₹{value / 1e3:.2f}K"
        else:
            return f"₹{value:.2f}"

In [17]:
# CELL 4: Example of creating and using the environment for each instrument in config
# You can iterate over your instruments and create individual environments
# (each environment will handle its own normalization for observations only).

envs = []
for instrument in config["instruments"]:
    env_config = {**config, **instrument}
    env = TradingEnv(env_config)
    envs.append(env)

# Example usage:
obs, info = envs[0].reset()
for _ in range(10):
    action = envs[0].action_space.sample()
    obs, reward, done, truncated, info = envs[0].step(action)
    if done:
        break

metrics = envs[0].get_metrics()
print(metrics)

# The above code demonstrates how you would create and run each environment.
# You can then wrap these environments in your RL training loop, stable_baselines, etc.

{'instrument': 'NIFTY_15M', 'initial_capital': '₹9.85L', 'final_capital': '₹9.20L', 'total_trades': 7, 'win_rate': '42.86%', 'total_reward': -1.57, 'max_drawdown': 0.0022943926032431548, 'profit_factor': 1.165809768637342}
